In [0]:
%sql
CREATE OR REPLACE TABLE workspace.idp.parsed_raw_documents AS
SELECT *,
ai_parse_document(content) AS parsed_content
FROM read_files('/Volumes/workspace/idp/final_project')

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.idp.classified_files AS
SELECT path, parsed_content,
ai_classify(CAST(parsed_content AS STRING), ARRAY('Receipt','Purchase Order','Invoice')) AS document_type
FROM workspace.idp.parsed_raw_documents

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.idp.invoices AS
SELECT
  invoice_details.Invoice,
  invoice_details.`Invoice Date` AS Invoice_Date,
  invoice_details.Terms,
  invoice_details.Currency,
  invoice_details.`Payment Method` AS Payment_Method,
  invoice_details.Total
FROM (
  SELECT 
    path,
    ai_extract(CAST(parsed_content AS STRING), 
      ARRAY('Invoice', 'Invoice Date', 'Terms', 'Currency', 'Payment Method', 'Total')
    ) AS invoice_details
  FROM workspace.idp.classified_files
  WHERE document_type = 'Invoice'
)

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.idp.purchase_orders AS
SELECT 
  po_details.`Buyer Name` AS Buyer_Name,
  po_details.`PO Number` AS PO_Number,
  po_details.Total
FROM (
  SELECT 
    path,
    ai_extract(CAST(parsed_content AS STRING), 
      ARRAY('Buyer Name', 'PO Number', 'Total')
    ) AS po_details
  FROM workspace.idp.classified_files
  WHERE document_type = 'Purchase Order'
)

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.idp.receipts AS
SELECT 
  receipt_details.`Receipt No` AS Receipt_No,
  receipt_details.Total
FROM (
  SELECT 
    path,
    ai_extract(CAST(parsed_content AS STRING), 
      ARRAY('Receipt No', 'Total')
    ) AS receipt_details
  FROM workspace.idp.classified_files
  WHERE document_type = 'Receipt'
)